In [3]:
!uv pip install selenium

Using Python 3.12.12 environment at: D:\projects\llm_engineering\.venv
Checked 1 package in 16ms


# Summarizing website using an open-source model(Ollama)

In [14]:
# Importing all the necessary modules
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup

### Web-scraping using Selenium and BeautifulSoup

In [ ]:
def fetch_the_content(url):
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service)

    driver.get(url)

    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.TAG_NAME, 'p'))
    )

    html = driver.page_source
    driver.quit()

    soup = BeautifulSoup(html, 'html.parser')

    for tag in soup(['script', 'style', 'nav', 'footer']):
        tag.decompose()

    paragraphs = soup.find_all('p')
    
    text = ''
    
    for p in paragraphs:
        content = p.get_text().strip()
        if content:
            text += content + '\n'

    return text[:3000]

    

In [16]:
# Loading the Api key, in this we do not need it while we are using ollama locally
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print('No Api key was found')
elif not api_key.startswith('sk-proj-'):
    print('An Api key was found, but it does not start with sk-proj-')
elif api_key.strip() != api_key:
    print('Api key was found, but it might have space or tab characters at the start or the end')
else:
    print('Api key found and it works!')

Api key found and it works!


In [17]:
# Setting system prompt and the user prompt
system_prompt = '''
Act as a snarky helper and help summarize the website briefly while adding
a bit of humour in your answer. Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
'''
user_prompt_prefix= '''
Provide a short summary of this website. Summarize
the content carefully also include news and announcements if any.

'''

In [18]:
# Getting the ollama api key and creating a python client library
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [19]:
!ollama pull llama3.2:1b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling 74701a8c35f6: 100% ▕██████████████████▏ 1.3 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 4f659a1e86d7: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 


In [ ]:
# Setting a message variable for API prompts
def messages_for(website):

    messages = [
        {'role': 'system', 'content':system_prompt},
        {'role': 'user', 'content': (user_prompt_prefix + website)}
    ]

    return messages

# Function to call the Api
def summarize_content(url):
    website= fetch_the_content(url)

    response = ollama.chat.completions.create(model='llama3.2:1b', messages= messages_for(website))

    return response.choices[0].message.content 
    

'Let\'s summarize this website in a nutshell:\n\n* Ed is the founder & CTO of Nebula.io, a company applying AI to help people discover their potential and pursue their passions.\n* He\'s also an avid computer programmer and music producer (who needs a degree for that?).\n* Ed loves experimenting with Large Language Models (LLMs) on his own terms - i.e. without taking over the world.\n\nAnnouncements:\n\n* Ed has mentioned that Nebula.io is expanding its services to tackle more complex challenges, implying some kind of AI-powered innovation.\n* There\'s a mention of an acquiring company "untapt" and getting acquired in 2021, which might be relevant news for fans of tech startups.\n* A link to Udemy courses where Ed has shared his expertise with the world, earning top ratings and thousands of enrollments.\n\nNote: Ed is essentially bragging about himself (and his AI work) without revealing much about Nebula.io\'s current projects or announcements. But hey, who needs transparency when you

In [ ]:
# Making it look a bit better in Markdown
def display_summary(url):
    summary = summarize_content(url)
    display(Markdown(summary))

display_summary("https://edwarddonner.com")


### Welcome to Edward's Lair (AKA Nebula.io)

*   Join the crew: Ed, the visionary behind Nebula.io. He's about AI for humanity, not just profits. He'd love to join in on some code sessions and possibly embarrass his friends over LLMs.
	+ Twitter Handle: @edwarddonner
	+ Website: [www.edwarddonner.com](http://www.edwarddonner.com) (not actually a website, but you get the idea)
*   **Recent News**: Nebula.io just acquired AI startup untapt in 2021. Talk about a merger... and possible talent drain.
	+ Recent Post on Hacker News: "Just made 10% more money than my old job for doing something I loved as an adult." Yes, it's a success story, but also a bit of a weird anecdote. Unpacking AI's potential is on the agenda here.

### What to Expect

*   **Talks**: If you're on Ed's radar, he'll likely chat tech, like LLMs (Large Language Models). But don't worry, we won't brainwash you into writing Python code.
	+ Education: 500,000+ students have taken his Udemy courses. Talk about a buzzword!
*   **Resources**: For those interested in diving deeper, Ed's curriculum is available on his website. It includes topics like:
	+ Introduction to LLMs
	+ Application areas (AI-assisted writing)
	+ Debunking common myths

In [23]:
# Trying some other websites
display_summary('https://en.wikipedia.org/wiki/Main_Page')

**Fishy Facts & World News**

* Ornithoprion (the fish with a bird-like skull and saw-like teeth) has made its last bow... or should I say, fin?!
* A great update on ancient history: April 24 is Armenian Genocide Remembrance Day (1915)
* Le Corbusier's impressive architectural feats have been recognized as a World Heritage Site
* We're not quite tarsiers yet - the Philippine tarsier (Carlito syrichta) remains on the endangered list

**News Highlights**

* Conservation efforts to protect ancient creatures are in full swing